# Using DSPy framework

Training DSPy on true/false biomix questions can lead to overfitting due to the high level of background knowledge (~100%). To distinguish between its biological knowledge and knowledge graph utilization, we will train it to generate correct Cypher statements instead.

To achieve this, we need an extended biomix test set that includes genes, diseases, and the number of relationships between genes and diseases in the knowledge graph. We will then ask the LLM to generate queries that output the correct number of results.

In [16]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd
import re

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")

Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


In [3]:
from langchain_community.graphs import Neo4jGraph

enhanced_graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
    enhanced_schema=True,
)

enhanced_schema = enhanced_graph.schema

print(enhanced_schema)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {name:apoc.any.property(rel, 'type'), count: apoc.any.property(rel, 'count')}] AS relationships"


Node properties:
- **Entity**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentargets.org/licence"
  - `version`: STRING Example: "22.11"
  - `id`: STRING Example: "obi:1110122"
  - `preferred_id`: STRING Example: "obi"
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `approvedSymbol`: STRING 
  - `biotype`: STRING 
- **ThingWithTaxon**
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentargets.org/licence"
  - `version`: STRING Example: "22.11"
  - `id`: STRING Example: "obi:1110122"
  - `preferred_id`: STRING Ex

In [5]:
# Cypher data extraction

def extract_cypher(message):
    text = message
    pattern = r"```cypher(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    try:
        return [match.strip() for match in matches]
    except Exception:
        raise ValueError(f"Failed to parse: {message}")

from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)

In [6]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

In [66]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        start = time.time()
        query = extract_cypher(llm_output)[0]
        result = run_with_timeout(query_cypher_graph, 60, graph, query)
        duration = time.time() - start
        return {
            "query":llm_output,
            "success":True,
            "result": list(result),
            "time": duration
        }
    except Exception as e:
        return {
            "query":llm_output,
            "result": f"Failed to execute query: {str(e)}",
            "success":False,
            "exception":str(e)
        }


In order to use DSPy prompt optimization capabilities we need a good evaluation dataset, so we'll use biomix for test and evaluation.

As a base query we will use an enchanced LLM schema that is provided by the LangChain.

In [8]:
# DSPy setup:

import dspy

llm = dspy.LM("openai/gpt-4o", max_tokens = 2000)
dspy.settings.configure(lm = llm)

d:\AppData\conda_envs\pistoia\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading biomix test-set

We need to load an extended version of BioMix test-set. It contains original disease names and gene names

In [28]:
questions_test = pd.read_excel("biomix_true_false_selected_augmented.xlsx")
questions_train = pd.read_excel("biomix_true_false_selected_augmented_2.xlsx")
questions_train['type'] = 'train'
questions_test['type'] = 'test'

questions = pd.concat([questions_train, questions_test])

We'll go over genes & diseases and query graph to retrieve the number of records. Results will be cached to speed things up. We'll do both exact (case-sensitive) and inexact (case-insensitive) queries. LLM result should lie between these two values in order to be considered correct

In [29]:
def run_gene_disease_query(gene, disease, substr=True):
    if substr:
        disease_query = f"LOWER(disease.name) CONTAINS LOWER('{disease}')"
    else:
        disease_query = f"LOWER(disease.name) = LOWER('{disease}')"
    query = f"""
MATCH (gene:HumanGene)-[:IS_PART_OF]-(assoc)-[:IS_PART_OF]-(disease:Disease)
WHERE gene.approvedSymbol = '{gene}'
  AND ({disease_query})
RETURN 
    gene.approvedSymbol as Gene,
    disease.name as Disease,
    head([label IN labels(assoc) WHERE label CONTAINS '.GeneToDiseaseAssociation']) as EvidenceType,
    assoc.score as Score,
    assoc.literature as Literature
ORDER BY assoc.score DESC
"""
    return list(graph.query(query))

In [32]:
# cache directories:

CACHE_DIR_EXACT = ".temp/gene-disease-exact"
CACHE_DIR_INEXACT = ".temp/gene-disease-inexact"

os.makedirs(CACHE_DIR_EXACT, exist_ok=True)
os.makedirs(CACHE_DIR_INEXACT, exist_ok=True)

count_exact = []
count_inexact = []

# iterating over the questions and running the queries, accumulating counts of results

for _, row in tqdm(questions.iterrows(), total=len(questions)):
    disease = row['s']
    gene = row['o']
    
    json_filename = os.path.join(f"{CACHE_DIR_EXACT}/{gene}-{disease}.json")
    if os.path.exists(json_filename):
        with open(json_filename) as f:
            res_exact = json.load(f)
    else:
        res_exact = run_gene_disease_query(gene, disease, substr=False)
        with open(json_filename, 'w') as f:
            json.dump(res_exact, f, indent=4)

    json_filename = os.path.join(f"{CACHE_DIR_INEXACT}/{gene}-{disease}.json")
    if os.path.exists(json_filename):
        with open(json_filename) as f:
            res_inexact = json.load(f)
    else:
        res_inexact = run_gene_disease_query(gene, disease, substr=True)
        with open(json_filename, 'w') as f:
            json.dump(res_inexact, f, indent=4)

    count_exact.append(len(res_exact))
    count_inexact.append(len(res_inexact))

questions['count_min'] = count_exact
questions['count_max'] = count_inexact
questions


100%|██████████| 199/199 [00:00<00:00, 204.71it/s]


,text,s,o,label,qid,rel,dir,s_type,o_type,p,q_type,type,count_min,count_max
0,Loeys-Dietz Syndrome associates Gene TGFBR1,Loeys-Dietz Syndrome,TGFBR1,True,110.0,disease_gene,direct,disease,gene,associated,tf,train,391,391
1,Congenital contractural arachnodactyly associa...,Congenital contractural arachnodactyly,FBN2,True,66.0,disease_gene,direct,disease,gene,associated,tf,train,1491,1491
2,Pseudoachondroplasia associates Gene COMP,Pseudoachondroplasia,COMP,True,290.0,disease_gene,direct,disease,gene,associated,tf,train,219,219
3,Fabry Disease associates Gene GLA,Fabry Disease,GLA,True,259.0,disease_gene,direct,disease,gene,associated,tf,train,1563,1563
4,Loeys-Dietz Syndrome associates Gene TGFBR2,Loeys-Dietz Syndrome,TGFBR2,True,195.0,disease_gene,direct,disease,gene,associated,tf,train,311,314
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Polycythemia Vera is associated with Gene RPS6KA3,Polycythemia Vera,RPS6KA3,False,NaN,disease_gene,direct,disease,gene,associated,tf,test,0,0
96,BIETTI CRYSTALLINE CORNEORETINAL DYSTROPHY is ...,BIETTI CRYSTALLINE CORNEORETINAL DYSTROPHY,RUNX2,False,NaN,disease_gene,direct,disease,gene,associated,tf,test,0,0
97,Pierson syndrome is associated with Gene GDF5,Pierson syndrome,GDF5,False,NaN,disease_gene,direct,disease,gene,associated,tf,test,0,0
98,Cystinuria is associated with Gene GHR,Cystinuria,GHR,False,NaN,disease_gene,direct,disease,gene,associated,tf,test,0,0


In [35]:
testset = [dspy.Example(statement=x['text'], answer=(x['count_min'], x['count_min'])).with_inputs("statement") for _, x in questions.iterrows() if x['type'] == 'test']
trainset = [dspy.Example(statement=x['text'], answer=(x['count_min'], x['count_min'])).with_inputs("statement") for _, x in questions.iterrows() if x['type'] == 'train']


[Example({'statement': 'Loeys-Dietz Syndrome associates Gene TGFBR1', 'answer': (391, 391)}) (input_keys={'statement'}),
 Example({'statement': 'Congenital contractural arachnodactyly associates Gene FBN2', 'answer': (1491, 1491)}) (input_keys={'statement'}),
 Example({'statement': 'Pseudoachondroplasia associates Gene COMP', 'answer': (219, 219)}) (input_keys={'statement'}),
 Example({'statement': 'Fabry Disease associates Gene GLA', 'answer': (1563, 1563)}) (input_keys={'statement'}),
 Example({'statement': 'Loeys-Dietz Syndrome associates Gene TGFBR2', 'answer': (311, 311)}) (input_keys={'statement'}),
 Example({'statement': 'DOYNE HONEYCOMB RETINAL DYSTROPHY associates Gene EFEMP1', 'answer': (70, 70)}) (input_keys={'statement'}),
 Example({'statement': 'Multiple Endocrine Neoplasia Type 2b associates Gene RET', 'answer': (398, 398)}) (input_keys={'statement'}),
 Example({'statement': 'Tuberous Sclerosis associates Gene TSC2', 'answer': (8693, 8693)}) (input_keys={'statement'}),
 E

## DSPy cypher query strategy

In [69]:
class GenerateCypher(dspy.Signature):
    """Generate cypher statement to check correctness of a statement"""
    statement = dspy.InputField()
    neo4j_schema = dspy.InputField(desc="Schema of a neo4j database")
    cypher = dspy.OutputField(desc="Valid cypher query that can be used to check correctness of the statement")


class GraphQueryCoT(dspy.Module):

    def __init__(self, schema):
        super().__init__()
        self.generate_query  = dspy.ChainOfThought(GenerateCypher)
        self.schema = schema
    
    def forward(self, statement):
        return self.generate_query(statement=statement, neo4j_schema=self.schema)

cot_cypher = GraphQueryCoT(enhanced_schema)

answer = cot_cypher(statement = "Polycythemia Vera is not associated with Gene JAK2")
answer


Prediction(
    reasoning='To verify the statement "Polycythemia Vera is not associated with Gene JAK2," we need to check if there is any association between the disease "Polycythemia Vera" and the gene "JAK2" in the database. The schema indicates that associations between genes and diseases are represented by the `GeneToDiseaseAssociation` node. We should look for a `GeneToDiseaseAssociation` relationship where the disease is "Polycythemia Vera" and the gene is "JAK2". The `Gene` node has an `approvedSymbol` property which can be used to identify the gene "JAK2". However, the schema does not explicitly list "Polycythemia Vera" as a disease name, so we will assume it is represented in one of the disease nodes. We will use a generic query to check for any such association.',
    cypher='```cypher\nMATCH (d:DiseaseOrPhenotypicFeature)-[:IS_PART_OF]->(a:GeneToDiseaseAssociation)<-[:IS_PART_OF]-(g:Gene)\nWHERE d.name = "Polycythemia Vera" AND g.approvedSymbol = "JAK2"\nRETURN a\n```'
)

In [74]:
def validate_query(example, pred, trace=None):
    res = query_graph(pred.cypher)

    if not res['success']:
        return False
    count = len(res['result'])
    
    if count >= example.answer[0] and count <= example.answer[1]:
        return True
    return False


In [75]:
from dspy.evaluate import Evaluate

fast_evaluate_program = Evaluate(devset = testset[:5], metric=validate_query, display_progress=True, display_table=10, provide_traceback=True)
evaluate_program = Evaluate(devset = testset, metric=validate_query, display_progress=True, display_table=10, provide_traceback=True)

Evaluating CoT before conducting MIPRO optimization

In [76]:
evaluate_program(cot_cypher)

Average Metric: 47.00 / 100 (47.0%): 100%|██████████| 100/100 [10:19<00:00,  6.19s/it]

2024/12/23 15:02:00 INFO dspy.evaluate.evaluate: Average Metric: 47 / 100 (47.0%)


,statement,answer,reasoning,cypher,validate_query
0,Polycythemia Vera is not associated with Gene JAK2,"(518, 518)","To verify the statement ""Polycythemia Vera is not associated with ...",```cypher\nMATCH (d:DiseaseOrPhenotypicFeature)-[:IS_PART_OF]->(a:...,
1,Cystic Fibrosis associates Gene CFTR,"(7134, 7134)","The statement ""Cystic Fibrosis associates Gene CFTR"" implies a rel...","```cypher\nMATCH (d:DiseaseOrPhenotypicFeature {name: ""Cystic Fibr...",
2,Cleidocranial Dysplasia associates Gene RUNX2,"(392, 392)","The statement ""Cleidocranial Dysplasia associates Gene RUNX2"" sugg...","```cypher\nMATCH (d:DiseaseOrPhenotypicFeature {name: ""Cleidocrani...",
3,Ellis-Van Creveld Syndrome associates Gene EVC2,"(1150, 1150)","The statement ""Ellis-Van Creveld Syndrome associates Gene EVC2"" su...","```cypher\nMATCH (d:DiseaseOrPhenotypicFeature {name: ""Ellis-Van C...",
4,Juvenile polyposis syndrome associates Gene BMPR1A,"(1154, 1154)","The statement ""Juvenile polyposis syndrome associates Gene BMPR1A""...","```cypher\nMATCH (d:DiseaseOrPhenotypicFeature {name: ""Juvenile po...",
5,Laron Syndrome associates Gene GHR,"(252, 252)","The statement ""Laron Syndrome associates Gene GHR"" suggests that t...",MATCH (d)-[:IS_PART_OF]->(a:GeneToDiseaseAssociation)<-[:IS_PART_O...,
6,Wiskott-Aldrich Syndrome is not associated with Gene WAS,"(545, 545)","To verify the statement ""Wiskott-Aldrich Syndrome is not associate...","MATCH (g:Gene {approvedSymbol: ""WAS""})-[:IS_PART_OF]->(a:GeneToDis...",
7,Smith-Lemli-Opitz Syndrome is not associated with Gene DHCR7,"(746, 746)","To verify the statement ""Smith-Lemli-Opitz Syndrome is not associa...","```cypher\nMATCH (d:DiseaseOrPhenotypicFeature {name: ""Smith-Lemli...",
8,Johanson-Blizzard syndrome associates Gene UBR1,"(53, 53)","The statement ""Johanson-Blizzard syndrome associates Gene UBR1"" im...","```cypher\nMATCH (d:DiseaseOrPhenotypicFeature {name: ""Johanson-Bl...",✔️ [True]
9,Noonan Syndrome associates Gene RAF1,"(209, 209)","The statement ""Noonan Syndrome associates Gene RAF1"" implies that ...","```cypher\nMATCH (d:DiseaseOrPhenotypicFeature {name: ""Noonan Synd...",


47.0

In [77]:
tp = dspy.MIPROv2(metric = validate_query, auto="light")
optimized_cot_cypher = tp.compile(cot_cypher, trainset=trainset)

2024/12/23 15:03:21 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 7
minibatch: True
num_candidates: 5
valset size: 79

2024/12/23 15:03:30 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2024/12/23 15:03:30 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2024/12/23 15:03:30 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=5 sets of demonstrations...


Bootstrapping set 1/5
Bootstrapping set 2/5
Bootstrapping set 3/5


100%|██████████| 20/20 [02:07<00:00,  6.39s/it]


Bootstrapped 2 full traces after 19 examples for up to 1 rounds, amounting to 20 attempts.
Bootstrapping set 4/5


100%|██████████| 20/20 [00:20<00:00,  1.01s/it]


...

2024/12/23 15:10:49 INFO dspy.evaluate.evaluate: Average Metric: 13 / 25 (52.0%)
2024/12/23 15:10:49 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 52.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 1'].
2024/12/23 15:10:49 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [48.0, 68.0, 80.0, 56.0, 80.0, 52.0]
2024/12/23 15:10:49 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [53.16]
2024/12/23 15:10:49 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 53.16
2024/12/23 15:10:49 INFO dspy.teleprompt.mipro_optimizer_v2: ===========================


2024/12/23 15:10:49 INFO dspy.teleprompt.mipro_optimizer_v2: == Minibatch Trial 7 / 7 ==



Average Metric: 18.00 / 25 (72.0%): 100%|██████████| 25/25 [00:35<00:00,  1.41s/it]

2024/12/23 15:11:24 INFO dspy.evaluate.evaluate: Average Metric: 18 / 25 (72.0%)
2024/12/23 15:11:24 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 72.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 4', 'Predictor 0: Few-Shot Set 4'].
2024/12/23 15:11:24 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [48.0, 68.0, 80.0, 56.0, 80.0, 52.0, 72.0]
2024/12/23 15:11:24 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [53.16]
2024/12/23 15:11:24 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 53.16
2024/12/23 15:11:24 INFO dspy.teleprompt.mipro_optimizer_v2: ===========================


2024/12/23 15:11:24 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Full Eval 1 =====
2024/12/23 15:11:24 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 80.0) from minibatch trials...



Average Metric: 51.00 / 79 (64.6%): 100%|██████████| 79/79 [01:04<00:00,  1.23it/s]

2024/12/23 15:12:29 INFO dspy.evaluate.evaluate: Average Metric: 51 / 79 (64.6%)
2024/12/23 15:12:29 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 64.56
2024/12/23 15:12:29 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [53.16, 64.56]
2024/12/23 15:12:29 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 64.56
2024/12/23 15:12:29 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2024/12/23 15:12:29 INFO dspy.teleprompt.mipro_optimizer_v2: 

2024/12/23 15:12:29 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 64.56!


In [78]:
evaluate_program(optimized_cot_cypher)

Average Metric: 59.00 / 100 (59.0%): 100%|██████████| 100/100 [14:12<00:00,  8.52s/it]

2024/12/23 15:27:21 INFO dspy.evaluate.evaluate: Average Metric: 59 / 100 (59.0%)


,statement,answer,reasoning,cypher,validate_query
0,Polycythemia Vera is not associated with Gene JAK2,"(518, 518)","To verify the statement ""Polycythemia Vera is not associated with ...","```cypher\nMATCH (d:Disease {name: ""Polycythemia Vera""})-[:IS_PART...",
1,Cystic Fibrosis associates Gene CFTR,"(7134, 7134)","To verify the statement ""Cystic Fibrosis associates Gene CFTR,"" we...","```cypher\nMATCH (d:Disease {name: ""Cystic Fibrosis""})-[:IS_PART_O...",
2,Cleidocranial Dysplasia associates Gene RUNX2,"(392, 392)","To verify the statement ""Cleidocranial Dysplasia associates Gene R...","```cypher\nMATCH (d:Disease {name: ""Cleidocranial Dysplasia""})-[:I...",
3,Ellis-Van Creveld Syndrome associates Gene EVC2,"(1150, 1150)","To verify the statement ""Ellis-Van Creveld Syndrome associates Gen...","```cypher\nMATCH (d:Disease {name: ""Ellis-Van Creveld Syndrome""})-...",
4,Juvenile polyposis syndrome associates Gene BMPR1A,"(1154, 1154)","To verify the statement ""Juvenile polyposis syndrome associates Ge...","```cypher\nMATCH (d:Disease {name: ""Juvenile polyposis syndrome""})...",
5,Laron Syndrome associates Gene GHR,"(252, 252)","To verify the statement ""Laron Syndrome associates Gene GHR,"" we n...","```cypher\nMATCH (d:DiseaseOrPhenotypicFeature {name: ""Laron Syndr...",
6,Wiskott-Aldrich Syndrome is not associated with Gene WAS,"(545, 545)","To verify the statement ""Wiskott-Aldrich Syndrome is not associate...","```cypher\nMATCH (d:Disease {name: ""Wiskott-Aldrich Syndrome""})-[:...",
7,Smith-Lemli-Opitz Syndrome is not associated with Gene DHCR7,"(746, 746)","To verify the statement ""Smith-Lemli-Opitz Syndrome is not associa...","```cypher\nMATCH (g:Gene {approvedSymbol: ""DHCR7""})-[:IS_PART_OF]-...",
8,Johanson-Blizzard syndrome associates Gene UBR1,"(53, 53)","To verify the statement ""Johanson-Blizzard syndrome associates Gen...","```cypher\nMATCH (d:DiseaseOrPhenotypicFeature {name: ""Johanson-Bl...",✔️ [True]
9,Noonan Syndrome associates Gene RAF1,"(209, 209)","To verify the statement ""Noonan Syndrome associates Gene RAF1"" usi...","```cypher\nMATCH (d:Disease {name: ""Noonan Syndrome""})-[:IS_PART_O...",


59.0

In [79]:
dspy.inspect_history(1)





[2024-12-23T15:27:20.848492]

System message:

Your input fields are:
1. `statement` (str)
2. `neo4j_schema` (str): Schema of a neo4j database

Your output fields are:
1. `reasoning` (str)
2. `cypher` (str): Valid cypher query that can be used to check correctness of the statement

All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## statement ## ]]
{statement}

[[ ## neo4j_schema ## ]]
{neo4j_schema}

[[ ## reasoning ## ]]
{reasoning}

[[ ## cypher ## ]]
{cypher}

[[ ## completed ## ]]

In adhering to this structure, your objective is: 
        Create a Cypher query to verify the accuracy of a genetic association statement between a disease and a gene. Utilize the provided Neo4j schema to identify and match the relevant nodes and relationships that represent the entities in the statement. Include a detailed reasoning process to explain the steps taken to construct the query. Return the generated Cypher query along with the reasoni

47% -> 59% is a progress, but we'd expect 100% on such a simple task. 

Can we utilize heavy MIPROv2?

In [81]:
tp_heavy = dspy.MIPROv2(metric = validate_query, auto="heavy")
optimized_cot_cypher_heavy = tp_heavy.compile(cot_cypher, trainset=trainset)

2024/12/23 15:31:29 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING HEAVY AUTO RUN SETTINGS:
num_trials: 50
minibatch: True
num_candidates: 38
valset size: 79

2024/12/23 15:31:33 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2024/12/23 15:31:33 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2024/12/23 15:31:33 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=38 sets of demonstrations...


Bootstrapping set 1/38
Bootstrapping set 2/38
Bootstrapping set 3/38


100%|██████████| 20/20 [00:18<00:00,  1.09it/s]


Bootstrapped 2 full traces after 19 examples for up to 1 rounds, amounting to 20 attempts.
Bootstrapping set 4/38


100%|██████████| 20/20 [00:18<00:00,  1.10it/s]


...

2024/12/23 16:31:21 INFO dspy.evaluate.evaluate: Average Metric: 18 / 25 (72.0%)
2024/12/23 16:31:21 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 72.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 36', 'Predictor 0: Few-Shot Set 19'].
2024/12/23 16:31:21 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [52.0, 56.0, 60.0, 64.0, 68.0, 64.0, 68.0, 68.0, 96.0, 80.0, 76.0, 64.0, 64.0, 52.0, 68.0, 92.0, 64.0, 64.0, 64.0, 64.0, 80.0, 60.0, 68.0, 64.0, 64.0, 72.0, 68.0, 48.0, 64.0, 72.0, 76.0, 60.0, 72.0, 52.0, 68.0, 60.0, 76.0, 68.0, 56.0, 52.0, 80.0, 64.0, 60.0, 64.0, 64.0, 80.0, 76.0, 80.0, 72.0]
2024/12/23 16:31:21 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [53.16, 72.15, 65.82, 70.89, 70.89]
2024/12/23 16:31:21 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 72.15
2024/12/23 16:31:21 INFO dspy.teleprompt.mipro_optimizer_v2: =============================


2024/12/23 16:31:21 INFO dspy.teleprompt.mipro_optimiz


Average Metric: 19.00 / 25 (76.0%): 100%|██████████| 25/25 [00:42<00:00,  1.69s/it]

2024/12/23 16:32:04 INFO dspy.evaluate.evaluate: Average Metric: 19 / 25 (76.0%)
2024/12/23 16:32:04 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 76.0 on minibatch of size 25 with parameters ['Predictor 0: Instruction 22', 'Predictor 0: Few-Shot Set 30'].
2024/12/23 16:32:04 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: [52.0, 56.0, 60.0, 64.0, 68.0, 64.0, 68.0, 68.0, 96.0, 80.0, 76.0, 64.0, 64.0, 52.0, 68.0, 92.0, 64.0, 64.0, 64.0, 64.0, 80.0, 60.0, 68.0, 64.0, 64.0, 72.0, 68.0, 48.0, 64.0, 72.0, 76.0, 60.0, 72.0, 52.0, 68.0, 60.0, 76.0, 68.0, 56.0, 52.0, 80.0, 64.0, 60.0, 64.0, 64.0, 80.0, 76.0, 80.0, 72.0, 76.0]
2024/12/23 16:32:04 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [53.16, 72.15, 65.82, 70.89, 70.89]
2024/12/23 16:32:04 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 72.15
2024/12/23 16:32:04 INFO dspy.teleprompt.mipro_optimizer_v2: =============================


2024/12/23 16:32:04 INFO dspy.teleprompt.mipro_o


Average Metric: 57.00 / 79 (72.2%): 100%|██████████| 79/79 [02:38<00:00,  2.00s/it]

2024/12/23 16:34:42 INFO dspy.evaluate.evaluate: Average Metric: 57 / 79 (72.2%)
2024/12/23 16:34:42 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [53.16, 72.15, 65.82, 70.89, 70.89, 72.15]
2024/12/23 16:34:42 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 72.15
2024/12/23 16:34:42 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2024/12/23 16:34:42 INFO dspy.teleprompt.mipro_optimizer_v2: 

2024/12/23 16:34:42 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 72.15!


In [82]:
evaluate_program(optimized_cot_cypher_heavy)

Average Metric: 67.00 / 100 (67.0%): 100%|██████████| 100/100 [24:00<00:00, 14.40s/it]

2024/12/23 17:05:20 INFO dspy.evaluate.evaluate: Average Metric: 67 / 100 (67.0%)


,statement,answer,reasoning,cypher,validate_query
0,Polycythemia Vera is not associated with Gene JAK2,"(518, 518)","The statement ""Polycythemia Vera is not associated with Gene JAK2""...",```cypher MATCH (d)-[:IS_PART_OF]->(a:GeneToDiseaseAssociation)<-[...,
1,Cystic Fibrosis associates Gene CFTR,"(7134, 7134)","The statement ""Cystic Fibrosis associates Gene CFTR"" implies that ...",```cypher MATCH (d)-[:IS_PART_OF]->(a:GeneToDiseaseAssociation)<-[...,
2,Cleidocranial Dysplasia associates Gene RUNX2,"(392, 392)","The statement ""Cleidocranial Dysplasia associates Gene RUNX2"" impl...",```cypher MATCH (d)-[:IS_PART_OF]->(a:GeneToDiseaseAssociation)<-[...,
3,Ellis-Van Creveld Syndrome associates Gene EVC2,"(1150, 1150)","The statement ""Ellis-Van Creveld Syndrome associates Gene EVC2"" im...",```cypher MATCH (d)-[:IS_PART_OF]->(a:GeneToDiseaseAssociation)<-[...,
4,Juvenile polyposis syndrome associates Gene BMPR1A,"(1154, 1154)","The statement ""Juvenile polyposis syndrome associates Gene BMPR1A""...",```cypher MATCH (d)-[:IS_PART_OF]->(a:GeneToDiseaseAssociation)<-[...,
5,Laron Syndrome associates Gene GHR,"(252, 252)","The statement ""Laron Syndrome associates Gene GHR"" implies a relat...",```cypher MATCH (d)-[:IS_PART_OF]->(a:GeneToDiseaseAssociation)<-[...,
6,Wiskott-Aldrich Syndrome is not associated with Gene WAS,"(545, 545)","The statement ""Wiskott-Aldrich Syndrome is not associated with Gen...",```cypher MATCH (d)-[:IS_PART_OF]->(a:GeneToDiseaseAssociation)<-[...,
7,Smith-Lemli-Opitz Syndrome is not associated with Gene DHCR7,"(746, 746)","The statement ""Smith-Lemli-Opitz Syndrome is not associated with G...",```cypher MATCH (d)-[:IS_PART_OF]->(a:GeneToDiseaseAssociation)<-[...,
8,Johanson-Blizzard syndrome associates Gene UBR1,"(53, 53)","The statement ""Johanson-Blizzard syndrome associates Gene UBR1"" im...",```cypher MATCH (d)-[:IS_PART_OF]->(a:GeneToDiseaseAssociation)<-[...,✔️ [True]
9,Noonan Syndrome associates Gene RAF1,"(209, 209)","The statement ""Noonan Syndrome associates Gene RAF1"" implies that ...",```cypher MATCH (d)-[:IS_PART_OF]->(a:GeneToDiseaseAssociation)<-[...,✔️ [True]


67.0

So: 
- default CoT: 47%
- "light" optimized CoT: 59%
- "heavy" optimized CoT: 67%

In [83]:
optimized_cot_cypher_heavy.save("09-optimized_cot_cypher_heavy-gpt4o.json", save_field_meta=True)
optimized_cot_cypher.save("09-optimized_cot_cypher-gpt4o.json", save_field_meta=True)